# Notebook 2 ? Erase Column

Default-nya membaca hasil dari Notebook 1 (`output/hasil_join.csv`). Anda juga bisa mengarahkan `INPUT_PATH` ke CSV lain. Pilih kolom yang ingin dipertahankan atau dibuang, lalu simpan sebagai file baru. File sumber tidak ditimpa. Semua kolom dibaca sebagai teks agar ID seperti `00123` tetap utuh.

In [ ]:
from pathlib import Path
import pandas as pd

# KONFIGURASI ? sesuaikan dengan file Anda
INPUT_PATH = Path("output/hasil_join.csv")  # Default: hasil dari Notebook 1; ubah jika sumbernya file lain
INPUT_SEP = ","  # Gunakan ";" jika pemisah CSV adalah titik koma

# Isi salah satu cara:
# KEEP_COLUMNS: hanya kolom yang tertulis dipertahankan (lebih aman untuk memilih sedikit kolom).
KEEP_COLUMNS = None  # Contoh: ["id", "nama", "kota"]
# DROP_COLUMNS: kolom yang tertulis akan dibuang.
DROP_COLUMNS = ["catatan_internal"]  # Contoh: ["kolom_lama", "catatan_internal"]

DROP_ALL_EMPTY_COLUMNS = False  # Jika True, buang kolom yang seluruh nilainya kosong
STRIP_TEXT = True
OUTPUT_PATH = Path("output/tabel_trim.csv")
OUTPUT_ENCODING = "utf-8-sig"
BASE_DIR = Path.cwd()


## Baca dan periksa kolom

In [ ]:
input_path = INPUT_PATH if INPUT_PATH.is_absolute() else BASE_DIR / INPUT_PATH
if not input_path.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {input_path}")

df = pd.read_csv(input_path, sep=INPUT_SEP, dtype="string", encoding="utf-8-sig")
df.columns = df.columns.astype(str).str.strip()
if STRIP_TEXT:
    for col in df.select_dtypes(include=["string", "object"]).columns:
        df[col] = df[col].str.strip()

print(f"File: {input_path}")
print(f"Ukuran awal: {len(df):,} baris x {len(df.columns)} kolom")
print("Kolom tersedia:")
for col in df.columns:
    print(f"- {col}")
display(df.head())


## Erase / trim kolom

Jika `KEEP_COLUMNS` diisi daftar kolom, hanya kolom tersebut yang disimpan. Jika bernilai `None`, kolom pada `DROP_COLUMNS` yang dibuang.

In [ ]:
if KEEP_COLUMNS is not None:
    missing = [col for col in KEEP_COLUMNS if col not in df.columns]
    if missing:
        raise KeyError(f"Kolom KEEP_COLUMNS tidak ditemukan: {missing}")
    result = df.loc[:, KEEP_COLUMNS].copy()
else:
    missing = [col for col in DROP_COLUMNS if col not in df.columns]
    if missing:
        print(f"Kolom DROP_COLUMNS tidak ditemukan (dilewati): {missing}")
    drop = [col for col in DROP_COLUMNS if col in df.columns]
    result = df.drop(columns=drop).copy()

if DROP_ALL_EMPTY_COLUMNS:
    empty_cols = [col for col in result.columns if result[col].isna().all()]
    result = result.drop(columns=empty_cols)
else:
    empty_cols = []

removed = [col for col in df.columns if col not in result.columns]
print(f"Kolom dibuang: {removed if removed else 'tidak ada'}")
print(f"Ukuran hasil: {len(result):,} baris x {len(result.columns)} kolom")
display(result.head())


## Simpan hasil trim

In [ ]:
output_path = OUTPUT_PATH if OUTPUT_PATH.is_absolute() else BASE_DIR / OUTPUT_PATH
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path, index=False, encoding=OUTPUT_ENCODING)
print(f"Tersimpan: {output_path.resolve()}")